# 03 — Feature Selection

**Project:** Supply Chain Analysis and Prediction  

## Purpose
Select the most predictive features using two independent methods — LASSO (L1 regularization) and RFE (Recursive Feature Elimination) — then form a consensus feature set for modelling.

## Why Feature Selection?
Even with only 15 features, selection helps:
- Remove noisy/redundant features that hurt generalization
- Identify which variables genuinely drive supply chain delays
- Improve model interpretability for business stakeholders

## Flow

| Step | Description |
|------|-------------|
| 1 | Load processed data |
| 2 | LASSO (LassoCV) — rank by absolute coefficient |
| 3 | RFE (GradientBoostingClassifier) — rank by recursive elimination |
| 4 | Consensus feature set — union of both methods |
| 5 | Model score comparison — before vs after selection |
| 6 | Save selected feature list |

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import RFE
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

from src.config import PROCESSED_CSV, IMAGES_DIR, REPORTS_DIR, TARGET_COL, PLOT_STYLE, FIGURE_DPI, RANDOM_STATE

plt.style.use(PLOT_STYLE)
pd.set_option('display.max_columns', None)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Processed Data

In [2]:
df = pd.read_csv(PROCESSED_CSV)
print(f'Processed shape: {df.shape}')
display(df.head())

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Keep only numeric
X = X.select_dtypes(include='number')
feature_names = list(X.columns)
print(f'\nFeatures ({len(feature_names)}): {feature_names}')
print(f'Target: {TARGET_COL} | Class balance: {y.value_counts().to_dict()}')

Processed shape: (50, 15)


,Product,Supplier,Warehouse_Location,Quantity,Unit_Price,Total_Cost,Logistics_Partner,Shipping_Method,Delivery_Status,Delay_Label,Cost_Per_Unit,Is_High_Value,Delivery_Month,Delivery_DayOfWeek,Delivery_Quarter
0,3,0,2,45,16.02,720.90,0,0,3,0,16.02,0,6,3,2
1,3,0,0,37,15.47,572.39,0,2,3,0,15.47,0,6,4,2
2,4,2,2,45,41.42,1863.90,3,3,0,1,41.42,1,6,6,2
3,0,1,0,53,9.60,508.80,0,1,0,1,9.60,0,6,4,2
4,2,4,0,68,29.13,1980.84,4,0,0,1,29.13,1,6,0,2



Features (14): ['Product', 'Supplier', 'Warehouse_Location', 'Quantity', 'Unit_Price', 'Total_Cost', 'Logistics_Partner', 'Shipping_Method', 'Delivery_Status', 'Cost_Per_Unit', 'Is_High_Value', 'Delivery_Month', 'Delivery_DayOfWeek', 'Delivery_Quarter']
Target: Delay_Label | Class balance: {0: 34, 1: 16}


## 2. LASSO — Rank Features by Absolute Coefficient

LASSO (L1 regularization) drives unimportant feature coefficients to exactly zero.  
Features with non-zero coefficients are selected.

In [3]:
# Scale features — LASSO is scale-sensitive
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# LassoCV — auto-selects best alpha via cross-validation
lasso = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=10000)
lasso.fit(X_scaled, y)

lasso_df = pd.DataFrame({
    'Feature': feature_names,
    'LASSO_Coefficient': lasso.coef_,
    'Abs_Coefficient': np.abs(lasso.coef_)
}).sort_values('Abs_Coefficient', ascending=False).reset_index(drop=True)

lasso_selected = lasso_df[lasso_df['Abs_Coefficient'] > 0]['Feature'].tolist()

print(f'Best alpha (LassoCV): {lasso.alpha_:.6f}')
print(f'Features selected by LASSO ({len(lasso_selected)}): {lasso_selected}')
print()
display(lasso_df.round(6))

Best alpha (LassoCV): 0.054731
Features selected by LASSO (2): ['Delivery_Status', 'Quantity']



,Feature,LASSO_Coefficient,Abs_Coefficient
0,Delivery_Status,-0.330955,0.330955
1,Quantity,0.003748,0.003748
2,Supplier,0.000000,0.000000
3,Warehouse_Location,0.000000,0.000000
4,Unit_Price,-0.000000,0.000000
5,Product,0.000000,0.000000
6,Total_Cost,0.000000,0.000000
7,Logistics_Partner,0.000000,0.000000
8,Shipping_Method,-0.000000,0.000000
9,Cost_Per_Unit,-0.000000,0.000000


In [4]:
# ── LASSO coefficient plot ──
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if c != 0 else 'lightgray' for c in lasso_df['LASSO_Coefficient']]
ax.barh(lasso_df['Feature'][::-1], lasso_df['LASSO_Coefficient'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('LASSO Coefficients (LassoCV)\nBlue = selected, Gray = zeroed out')
ax.set_xlabel('Coefficient')
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03a_lasso_coefficients.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03a_lasso_coefficients.png')

Plot saved -> images/03a_lasso_coefficients.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7024\2728390753.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. RFE — Recursive Feature Elimination

RFE iteratively removes the least important feature, ranking all features by the order they were eliminated.  
Using GradientBoostingClassifier as the estimator.

In [5]:
# RFE with GradientBoostingClassifier
# Select top half of features
n_select = max(3, len(feature_names) // 2)

gb = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
rfe = RFE(estimator=gb, n_features_to_select=n_select, step=1)
rfe.fit(X, y)

rfe_df = pd.DataFrame({
    'Feature': feature_names,
    'RFE_Rank': rfe.ranking_,
    'RFE_Selected': rfe.support_
}).sort_values('RFE_Rank').reset_index(drop=True)

rfe_selected = rfe_df[rfe_df['RFE_Selected']]['Feature'].tolist()

print(f'Features selected by RFE (top {n_select}): {rfe_selected}')
display(rfe_df)

Features selected by RFE (top 7): ['Shipping_Method', 'Is_High_Value', 'Delivery_Quarter', 'Delivery_DayOfWeek', 'Delivery_Month', 'Delivery_Status', 'Cost_Per_Unit']


,Feature,RFE_Rank,RFE_Selected
0,Shipping_Method,1,True
1,Is_High_Value,1,True
2,Delivery_Quarter,1,True
3,Delivery_DayOfWeek,1,True
4,Delivery_Month,1,True
5,Delivery_Status,1,True
6,Cost_Per_Unit,1,True
7,Logistics_Partner,2,False
8,Total_Cost,3,False
9,Unit_Price,4,False


In [6]:
# ── RFE ranking plot ──
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if sel else 'lightcoral' for sel in rfe_df['RFE_Selected']]
ax.barh(rfe_df['Feature'][::-1], rfe_df['RFE_Rank'][::-1], color=colors[::-1])
ax.set_title('RFE Feature Ranking\nBlue = selected (rank 1), Red = eliminated')
ax.set_xlabel('Rank (1 = most important)')
ax.axvline(1.5, color='black', linewidth=0.8, linestyle='--', label='Selection boundary')
ax.legend()
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03b_rfe_ranking.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03b_rfe_ranking.png')

Plot saved -> images/03b_rfe_ranking.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7024\2639212791.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Consensus Feature Set

In [7]:
# Union of LASSO + RFE selections
consensus = sorted(set(lasso_selected) | set(rfe_selected))

# If LASSO zeroed everything (can happen on tiny datasets), fall back to RFE only
if not lasso_selected:
    consensus = rfe_selected
    print('Note: LASSO zeroed all features (small dataset) — using RFE selection only.')

consensus_df = pd.DataFrame({
    'Feature': consensus,
    'In_LASSO': [f in lasso_selected for f in consensus],
    'In_RFE':   [f in rfe_selected   for f in consensus],
})
consensus_df['In_Both'] = consensus_df['In_LASSO'] & consensus_df['In_RFE']

print(f'LASSO selected  : {len(lasso_selected)} features')
print(f'RFE selected    : {len(rfe_selected)} features')
print(f'Consensus (union): {len(consensus)} features')
print(f'In both methods : {consensus_df["In_Both"].sum()} features')
print()
display(consensus_df)

LASSO selected  : 2 features
RFE selected    : 7 features
Consensus (union): 8 features
In both methods : 1 features



,Feature,In_LASSO,In_RFE,In_Both
0,Cost_Per_Unit,False,True,False
1,Delivery_DayOfWeek,False,True,False
2,Delivery_Month,False,True,False
3,Delivery_Quarter,False,True,False
4,Delivery_Status,True,True,True
5,Is_High_Value,False,True,False
6,Quantity,True,False,False
7,Shipping_Method,False,True,False


In [8]:
# ── Venn-style comparison bar chart ──
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(consensus))
width = 0.35
lasso_vals = [1 if f in lasso_selected else 0 for f in consensus]
rfe_vals   = [1 if f in rfe_selected   else 0 for f in consensus]

ax.bar(x - width/2, lasso_vals, width, label='LASSO', color='steelblue', alpha=0.8)
ax.bar(x + width/2, rfe_vals,   width, label='RFE',   color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(consensus, rotation=35, ha='right')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Not Selected', 'Selected'])
ax.set_title('Feature Selection Comparison: LASSO vs RFE')
ax.legend()
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03c_feature_selection_comparison.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03c_feature_selection_comparison.png')

Plot saved -> images/03c_feature_selection_comparison.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7024\414346642.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Model Score — Before vs After Feature Selection

In [9]:
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# All features
cv_all = cross_val_score(rf, X, y, cv=5, scoring='accuracy')

# Consensus features only
X_consensus = X[consensus]
cv_sel = cross_val_score(rf, X_consensus, y, cv=5, scoring='accuracy')

comparison = pd.DataFrame({
    'Feature Set': [f'All features ({len(feature_names)})', f'Selected features ({len(consensus)})'],
    'CV Accuracy (mean)': [cv_all.mean().round(4), cv_sel.mean().round(4)],
    'CV Accuracy (std)':  [cv_all.std().round(4),  cv_sel.std().round(4)],
})

print('Model Score — Before vs After Feature Selection (RandomForest, 5-fold CV):')
display(comparison)

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(comparison['Feature Set'], comparison['CV Accuracy (mean)'],
              color=['steelblue', 'mediumseagreen'], edgecolor='white',
              yerr=comparison['CV Accuracy (std)'], capsize=5)
ax.set_title('CV Accuracy: All Features vs Selected Features')
ax.set_ylabel('Mean CV Accuracy')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, comparison['CV Accuracy (mean)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03d_before_after_selection.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03d_before_after_selection.png')

Model Score — Before vs After Feature Selection (RandomForest, 5-fold CV):


,Feature Set,CV Accuracy (mean),CV Accuracy (std)
0,All features (14),1.0,0.0
1,Selected features (8),1.0,0.0


Plot saved -> images/03d_before_after_selection.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7024\1252138235.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Save Selected Feature List

In [10]:
selected_path = REPORTS_DIR / 'selected_features.csv'
consensus_df.to_csv(selected_path, index=False)
print(f'Selected features saved -> {selected_path}')

# Save processed data with only selected features + target
df_selected = df[consensus + [TARGET_COL]]
sel_data_path = REPORTS_DIR / 'data_selected_features.csv'
df_selected.to_csv(sel_data_path, index=False)
print(f'Selected feature dataset saved -> {sel_data_path}')
print(f'Final dataset shape: {df_selected.shape}')

Selected features saved -> D:\automation\Supply chain Analysis and Prediction\reports\selected_features.csv
Selected feature dataset saved -> D:\automation\Supply chain Analysis and Prediction\reports\data_selected_features.csv
Final dataset shape: (50, 9)


## Feature Selection Summary

| Method | Features Selected | Approach |
|---|---|---|
| LASSO (LassoCV) | Non-zero coefficient features | L1 regularization drives noise to zero |
| RFE (GradientBoosting) | Top 50% of features | Recursive elimination by importance |
| **Consensus (Union)** | **Union of both** | Final feature set for modelling |

**Key insight:** Features selected by both methods are the strongest predictors of supply chain delays.

---

**Next step:** `04_Model_Training.ipynb` — train 5 classification models on the consensus feature set.